# Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.sparse import hstack, csr_matrix

# --- Configuration ---
PROCESSED_DATA_DIR = Path("data/02_processed")
FEATURES_DATA_DIR = Path("data/03_features")
CLEAN_DATA_FILE = PROCESSED_DATA_DIR / "clean_cve_data.csv"
SBERT_MODEL_NAME = 'all-MiniLM-L6-v2'

FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Setup complete. Paths and libraries are ready.")

In [ ]:
print("Loading clean data...")
df = pd.read_csv(CLEAN_DATA_FILE)
df.head()

In [ ]:
print(f"Generating text embeddings with '{SBERT_MODEL_NAME}'...")
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

descriptions = df["Description"].tolist()
text_embeddings = sbert_model.encode(descriptions, show_progress_bar=True)

print(f"Text embeddings created with shape: {text_embeddings.shape}")

In [ ]:
print("Generating categorical features for CWE...")
cwe_encoder = OneHotEncoder(handle_unknown='ignore')
cwe_features = cwe_encoder.fit_transform(df[["CWE"]])
print(f"CWE features created with shape: {cwe_features.shape}")

In [ ]:
print("Generating numeric features for CVSS Score...")
scaler = StandardScaler()
numeric_features = scaler.fit_transform(df[["CVSS_Score"]])
print(f"Numeric features created with shape: {numeric_features.shape}")

In [ ]:
print("Combining all features into a final matrix...")
X = np.hstack([
    text_embeddings,
    cwe_features.toarray(),
    numeric_features
])

y = df["Severity"]

print(f"Final feature matrix `X` created with shape: {X.shape}")
print(f"Target variable `y` created with shape: {y.shape}")

In [ ]:
print("Saving features and preprocessing objects...")
joblib.dump((X, y), FEATURES_DATA_DIR / "features.pkl")
joblib.dump(cwe_encoder, FEATURES_DATA_DIR / "cwe_encoder.pkl")
joblib.dump(scaler, FEATURES_DATA_DIR / "scaler.pkl")

print("\nFeature engineering complete!")